# Multimodal Service Index (SI) — Luxembourg

Computing **SI**, the level of PT service accessible from each origin, accounting for both direct walking access and bike-sharing enabled ac- cess (Fan & Harper, 2024).

---

## Folder structure

Place all input files in a `data/` folder next to this notebook:

```
Phase 1/
├── SI_accesibility.ipynb        ← this notebook
├── requirements.txt
├── data/
│   ├── EU_DEM_Lux.tif          ← DEM raster (GeoTIFF)
│   ├── origins.csv             ← population grid origins
│   ├── stops_only_lux.csv      ← PT stops clipped to Luxembourg boundary
│   └── bike_stations_merged.csv← SBBS bike-share stations
└── cache/                      ← auto-created; stores GraphML + output CSVs
```

## Required columns per input file

| File | Required columns | Notes |
|---|---|---|
| `origins.csv` | `GRD_ID`, `x`, `y` | - |
| `stops_only_luxemburg.csv` | `stop_id`, `Fj`, `lon`, `lat` | `Fj` = arrivals/hour; lon/lat in WGS84 |
| `bike_stations_merged.csv` | `id`, `longitude`, `latitude` | - |

## Installation

```bash
pip install -r requirements.txt
```

## Outputs

Graphs are cached to `cache/` as GraphML after the first download — subsequent runs skip the download.

All CSVs are written to `cache/`:
- `SI_results.csv` — baseline SI per origin
- `sensitivity_thresholds/` — frequency-weighted sensitivity scenarios
- `sensitivity_binary/` — binary Fj sensitivity scenarios
- `sensitivity_combined_summary.csv` — combined summary table


In [ ]:
import os
import math
import time
import pandas as pd
import re
import numpy as np
import networkx as nx
import geopandas as gpd
import osmnx as ox
from pathlib import Path
from shapely.geometry import Point

In [ ]:
# =========================
# SETTINGS
# =========================

# --- Project root ---
PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
CACHE_DIR    = PROJECT_ROOT / "cache"

# --- Define study area ---
USE_PLACE_QUERY = True
PLACE_NAME = {"country": "Luxembourg"}

# --- CRS ---
PROJECT_CRS = "EPSG:2169"

# --- DEM path (GeoTIFF) ---
DEM_PATH = str(DATA_DIR / "EU_DEM_Lux.tif")


# --- Input files ---
ORIGINS_CSV    = str(DATA_DIR / "origins.csv")
ORIGINS_ID_COL  = "GRD_ID"
ORIGINS_LON_COL = None
ORIGINS_LAT_COL = None
ORIGINS_X_COL   = "x"
ORIGINS_Y_COL   = "y"

STOPS_CSV     = str(DATA_DIR / "stops_only_luxemburg.csv")
STOP_ID_COL   = "stop_id"
STOP_FJ_COL   = "Fj"
STOP_LON_COL  = "lon"
STOP_LAT_COL  = "lat"
STOP_X_COL    = None
STOP_Y_COL    = None

STATIONS_CSV    = str(DATA_DIR / "bike_stations_merged.csv")
STATION_ID_COL  = "id"
STATION_LON_COL = "longitude"
STATION_LAT_COL = "latitude"
STATION_X_COL   = None
STATION_Y_COL   = None

# --- Model thresholds (minutes) ---
WALK_CUTOFF = 8
BIKE_CUTOFF = 12

# --- Walking impedance: Tobler + steps ---
STAIR_SPEED_M_PER_MIN = 42.0

# --- Bike impedance: e-bike rule ---
BASE_KMH             = 18.0
MIN_KMH              = 6.0
UPHILL_THRESHOLD_PCT = 3.0
PENALTY_PER_PCT      = 1.5
DOWNHILL_CAP_KMH     = 24.0

# --- Cache / output dirs ---
WALK_GRAPHML = str(CACHE_DIR / "lux_walk.graphml")
BIKE_GRAPHML = str(CACHE_DIR / "lux_bike.graphml")

# --- OSMnx settings ---
ox.settings.use_cache           = True
ox.settings.log_console         = True
ox.settings.requests_timeout    = 180
ox.settings.overpass_rate_limit = True

print("All good.")


All good.


In [ ]:
# =========================
# HELPERS
# =========================

def ensure_dir(path) -> None:
    os.makedirs(path, exist_ok=True)


def tobler_speed_kmh(grade_dec: float) -> float:
    """Tobler hiking function: v = 6 * exp(-3.5 * |S + 0.05|)"""
    return 6.0 * math.exp(-3.5 * abs(grade_dec + 0.05))


def ebike_speed_kmh(grade_dec: float) -> float:
    """E-bike speed model with uphill penalty and capped downhill benefit."""
    g_pct = grade_dec * 100.0
    if g_pct >= 0:
        excess = max(0.0, g_pct - UPHILL_THRESHOLD_PCT)
        speed = BASE_KMH - PENALTY_PER_PCT * excess
        return max(MIN_KMH, speed)
    else:
        benefit = 1.0 + 0.02 * min(10.0, abs(g_pct))
        return min(DOWNHILL_CAP_KMH, BASE_KMH * benefit)


def gdf_from_csv_points(
    csv_path: str,
    id_col: str,
    lon_col: str | None,
    lat_col: str | None,
    x_col: str | None,
    y_col: str | None,
    crs_out: str,
) -> gpd.GeoDataFrame:
    """Read a CSV with point coordinates and return a projected GeoDataFrame."""
    df = pd.read_csv(csv_path)
    if lon_col and lat_col:
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
            crs="EPSG:4326",
        ).to_crs(crs_out)
    elif x_col and y_col:
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[x_col], df[y_col]),
            crs=crs_out,
        )
    else:
        raise ValueError(f"{csv_path}: Provide either (lon_col,lat_col) or (x_col,y_col).")

    if id_col not in gdf.columns:
        raise ValueError(f"{csv_path}: Missing required id column '{id_col}'")
    return gdf


def drop_bad_edges_inplace(G: nx.MultiDiGraph, time_attr: str) -> None:
    """Remove edges where time_attr is missing / None / NaN — in-place."""
    bad = []
    for u, v, k, d in G.edges(keys=True, data=True):
        t = d.get(time_attr, None)
        if t is None or (isinstance(t, float) and math.isnan(t)):
            bad.append((u, v, k))
    G.remove_edges_from(bad)
    print(f"  Dropped {len(bad)} edges with missing '{time_attr}'")


def snap_points(
    gdf: gpd.GeoDataFrame,
    G,
    label: str,
) -> pd.Series:
    """Snap points to the nearest graph node. All points are snapped regardless of distance."""
    nearest_nodes = ox.distance.nearest_nodes(G, X=gdf.geometry.x, Y=gdf.geometry.y)
    nearest_series = pd.Series(nearest_nodes, index=gdf.index)
    print(f"  [{label}]: snapped {len(nearest_series)} points.")
    return nearest_series


def compute_si_for_origins(
    origin_walk_node: dict,
    G_walk,
    walk_cutoff: float,
    node_to_stops: dict,
    walknode_to_station_ids: dict,
    station_bike_reach: dict,
    Fj_dict: dict,
) -> pd.DataFrame:
    """
    Core SI computation extracted into a reusable function.
    Returns a DataFrame with columns: origin_id, SI_walk, SI_bike_added, SI_total.
    """
    rows = []
    missing = 0
    for oid, onode in origin_walk_node.items():
        if onode not in G_walk:
            missing += 1
            rows.append([oid, 0.0, 0.0, 0.0])
            continue

        distw = nx.single_source_dijkstra_path_length(
            G_walk, onode, cutoff=walk_cutoff, weight="walk_time_min"
        )

        # W(i): walk-reachable stops
        W = set()
        for n in distw:
            if n in node_to_stops:
                W.update(node_to_stops[n])

        # S(i): walk-reachable SBBS stations
        S = set()
        for n in distw:
            if n in walknode_to_station_ids:
                S |= walknode_to_station_ids[n]

        # B(i): union of bike-reachable stops from those stations
        B = set()
        for st_id in S:
            B |= station_bike_reach.get(st_id, set())

        added = B - W

        SI_walk       = sum(Fj_dict[sid] for sid in W)
        SI_bike_added = sum(Fj_dict[sid] for sid in added)
        SI_total      = SI_walk + SI_bike_added

        rows.append([oid, SI_walk, SI_bike_added, SI_total])

    if missing:
        print(f"  Origins missing in walk graph: {missing}/{len(origin_walk_node)}")
    return pd.DataFrame(rows, columns=["origin_id", "SI_walk", "SI_bike_added", "SI_total"])


def cache_station_bike_reach(
    station_bike_node: dict,
    G_bike,
    bike_cutoff: float,
    node_to_stops_bike: dict,
) -> dict:
    """Compute station -> set of bike-reachable stop_ids within cutoff."""
    result = {}
    missing = 0
    for st_id, bnode in station_bike_node.items():
        if bnode not in G_bike:
            missing += 1
            result[st_id] = set()
            continue
        dist = nx.single_source_dijkstra_path_length(
            G_bike, bnode, cutoff=bike_cutoff, weight="bike_time_min"
        )
        reachable = set()
        for n in dist:
            if n in node_to_stops_bike:
                reachable.update(node_to_stops_bike[n])
        result[st_id] = reachable
    if missing:
        print(f"  Stations missing in bike graph: {missing}/{len(station_bike_node)}")
    return result


print("All good.")


All good.


In [ ]:
# =========================
# 1) LOAD / DOWNLOAD GRAPHS
# =========================

ensure_dir(CACHE_DIR)
ensure_dir(str(CACHE_DIR / "sensitivity_thresholds"))
ensure_dir(str(CACHE_DIR / "sensitivity_binary"))

if os.path.exists(WALK_GRAPHML) and os.path.exists(BIKE_GRAPHML):
    print("Loading cached graphs from GraphML...")
    G_walk = ox.load_graphml(WALK_GRAPHML)
    G_bike = ox.load_graphml(BIKE_GRAPHML)

else:
    print("Downloading OSM graph with OSMnx...")

    custom_filter = (
        '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|secondary|'
        'tertiary|unclassified|residential|living_street|service|cycleway|path|'
        'footway|pedestrian|steps"]'
        '["area"!~"yes"]'
    )

    if USE_PLACE_QUERY:
        G_all = ox.graph_from_place(
            PLACE_NAME, custom_filter=custom_filter, simplify=True, retain_all=False
        )
    else:
        G_all = ox.graph_from_bbox(
            north, south, east, west, custom_filter=custom_filter,
            simplify=True, retain_all=False,
        )

    # ---- WALK GRAPH ----
    G_walk = G_all.copy()
    walk_remove = []
    WALK_EXCLUDE_HW = {"motorway", "motorway_link", "trunk", "trunk_link"}
    for u, v, k, d in G_walk.edges(keys=True, data=True):
        hw = d.get("highway", "")
        hw_set = set(hw) if isinstance(hw, list) else {hw}
        foot = d.get("foot", None)
        if hw_set & WALK_EXCLUDE_HW:
            walk_remove.append((u, v, k))
        elif foot == "no":
            walk_remove.append((u, v, k))
    G_walk.remove_edges_from(walk_remove)
    print(f"Walk graph: removed {len(walk_remove)} edges (motorway/trunk, foot=no)")

    # ---- BIKE GRAPH ----
    G_bike = G_all.copy()
    bike_remove = []
    BIKE_EXCLUDE_HW = {"motorway", "motorway_link", "trunk", "trunk_link"}
    for u, v, k, d in G_bike.edges(keys=True, data=True):
        hw = d.get("highway", "")
        hw_set = set(hw) if isinstance(hw, list) else {hw}
        bicycle = d.get("bicycle", None)
        is_steps = "steps" in hw_set
        if hw_set & BIKE_EXCLUDE_HW:
            bike_remove.append((u, v, k))
        elif is_steps:
            bike_remove.append((u, v, k))
        elif bicycle == "no":
            bike_remove.append((u, v, k))
    G_bike.remove_edges_from(bike_remove)
    print(f"Bike graph: removed {len(bike_remove)} edges (motorway/trunk, steps, bicycle=no)")

    # ---- ADD ELEVATION + GRADES (before projecting) ----
    print("Adding node elevations from DEM...")
    G_walk = ox.elevation.add_node_elevations_raster(G_walk, DEM_PATH, cpus=1)
    G_bike = ox.elevation.add_node_elevations_raster(G_bike, DEM_PATH, cpus=1)
    print("Computing edge grades...")
    G_walk = ox.elevation.add_edge_grades(G_walk)
    G_bike = ox.elevation.add_edge_grades(G_bike)
    print("Elevation and grades added.")

    # ---- PROJECT + SAVE  ----
    G_walk = ox.project_graph(G_walk, to_crs=PROJECT_CRS)
    G_bike = ox.project_graph(G_bike, to_crs=PROJECT_CRS)
    ensure_dir(CACHE_DIR)
    ox.save_graphml(G_walk, WALK_GRAPHML)
    ox.save_graphml(G_bike, BIKE_GRAPHML)
    print("Graphs saved to:", WALK_GRAPHML, "and", BIKE_GRAPHML)

# Ensure projected
if G_walk.graph.get("crs") != PROJECT_CRS:
    G_walk = ox.project_graph(G_walk, to_crs=PROJECT_CRS)
if G_bike.graph.get("crs") != PROJECT_CRS:
    G_bike = ox.project_graph(G_bike, to_crs=PROJECT_CRS)

# Convert walk to undirected
G_walk = ox.convert.to_undirected(G_walk)

# Keep only largest connected component
# (removes orphan nodes and tiny disconnected fragments from OSM data)
print("Cleaning disconnected components...")

walk_components = list(nx.connected_components(G_walk))
largest_walk = max(walk_components, key=len)
print(f"  Walk: {len(walk_components)} components, largest has {len(largest_walk)} nodes, "
      f"dropping {G_walk.number_of_nodes() - len(largest_walk)} orphan nodes")
G_walk = G_walk.subgraph(largest_walk).copy()

bike_components = list(nx.weakly_connected_components(G_bike))
largest_bike = max(bike_components, key=len)
print(f"  Bike: {len(bike_components)} components, largest has {len(largest_bike)} nodes, "
      f"dropping {G_bike.number_of_nodes() - len(largest_bike)} orphan nodes")
G_bike = G_bike.subgraph(largest_bike).copy()

print(f"\nWalk graph: {G_walk.number_of_nodes()} nodes, {G_walk.number_of_edges()} edges")
print(f"Bike graph: {G_bike.number_of_nodes()} nodes, {G_bike.number_of_edges()} edges")
print(f"Avg edges/node (walk): {G_walk.number_of_edges() / G_walk.number_of_nodes():.2f}")
print(f"Avg edges/node (bike): {G_bike.number_of_edges() / G_bike.number_of_nodes():.2f}")

Walk graph: removed 893 edges (motorway/trunk, foot=no)
Bike graph: removed 8663 edges (motorway/trunk, steps, bicycle=no)
Adding node elevations from DEM...
Computing edge grades...
Elevation and grades added.
Graphs saved to: /Users/fgsumer/Desktop/thesis_code/Phase 1/cache/lux_walk.graphml and /Users/fgsumer/Desktop/thesis_code/Phase 1/cache/lux_bike.graphml
Cleaning disconnected components...
  Walk: 447 components, largest has 98560 nodes, dropping 729 orphan nodes
  Bike: 1822 components, largest has 96231 nodes, dropping 3058 orphan nodes

Walk graph: 98560 nodes, 134364 edges
Bike graph: 96231 nodes, 243666 edges
Avg edges/node (walk): 1.36
Avg edges/node (bike): 2.53


In [ ]:
import numpy as np

def check_elevation_grades(G, name):
    elevations = [d.get('elevation') for _, d in G.nodes(data=True)]
    missing_elev = sum(1 for e in elevations if e is None)
    valid_elev = [e for e in elevations if e is not None]
    
    grades = [d.get('grade') for _, _, d in G.edges(data=True)]
    missing_grade = sum(1 for g in grades if g is None)
    valid_grades = [g for g in grades if g is not None]
    
    print(f"\n{'='*40}")
    print(f"Graph: {name}")
    print(f"  Nodes: {G.number_of_nodes()} | Missing elevation: {missing_elev}")
    print(f"  Edges: {G.number_of_edges()} | Missing grade: {missing_grade}")
    if valid_elev:
        print(f"  Elevation range: {min(valid_elev):.1f}m – {max(valid_elev):.1f}m")
    if valid_grades:
        print(f"  Grade range: {min(valid_grades)*100:.1f}% – {max(valid_grades)*100:.1f}%")
        print(f"  Mean abs grade: {np.mean(np.abs(valid_grades))*100:.2f}%")

check_elevation_grades(G_walk, "Walk")
check_elevation_grades(G_bike, "Bike")


Graph: Walk
  Nodes: 98560 | Missing elevation: 0
  Edges: 134364 | Missing grade: 0
  Elevation range: 132.0m – 551.0m
  Grade range: -596.8% – 479.7%
  Mean abs grade: 4.86%

Graph: Bike
  Nodes: 96231 | Missing elevation: 0
  Edges: 243666 | Missing grade: 0
  Elevation range: 132.0m – 551.0m
  Grade range: -535.1% – 535.1%
  Mean abs grade: 4.72%


In [ ]:
# =========================
# 3) COMPUTE EDGE TRAVEL TIMES
# =========================

# Clip grades to physically plausible and
# artifacts on edges shorter than the DEM cell size (25 m).
GRADE_CAP = 0.40   # ±40% covers the steepest real pedestrian/e-bike infrastructure


def safe_grade(raw) -> float:
    """Return a finite grade clipped to ±GRADE_CAP; treat None/NaN as 0.0."""
    if raw is None:
        return 0.0
    try:
        g = float(raw)
    except (TypeError, ValueError):
        return 0.0
    if math.isnan(g):
        return 0.0
    return max(-GRADE_CAP, min(GRADE_CAP, g))


print("Computing walk_time_min on edges...")
for u, v, k, d in G_walk.edges(keys=True, data=True):
    L = float(d.get("length", 0.0))
    grade = safe_grade(d.get("grade"))
    hw = d.get("highway", None)
    is_steps = ("steps" in hw) if isinstance(hw, list) else (hw == "steps")

    if L <= 0:
        d["walk_time_min"] = None
        continue
    if is_steps:
        d["walk_time_min"] = L / STAIR_SPEED_M_PER_MIN
    else:
        speed_kmh = tobler_speed_kmh(grade)
        d["walk_time_min"] = ((L / 1000.0) / speed_kmh) * 60.0

print("Computing bike_time_min on edges...")
for u, v, k, d in G_bike.edges(keys=True, data=True):
    L = float(d.get("length", 0.0))
    grade = safe_grade(d.get("grade"))

    if L <= 0:
        d["bike_time_min"] = None
        continue
    speed_kmh = ebike_speed_kmh(grade)
    d["bike_time_min"] = ((L / 1000.0) / speed_kmh) * 60.0

drop_bad_edges_inplace(G_walk, "walk_time_min")
drop_bad_edges_inplace(G_bike, "bike_time_min")

print("All good with travel times.")

Computing walk_time_min on edges...
Computing bike_time_min on edges...
  Dropped 0 edges with missing 'walk_time_min'
  Dropped 0 edges with missing 'bike_time_min'
All good with travel times.


In [ ]:
# =========================
# 4) LOAD POINT DATA + SNAP TO NODES
# =========================

print("Loading points...")
orig_gdf = gdf_from_csv_points(
    ORIGINS_CSV, ORIGINS_ID_COL,
    ORIGINS_LON_COL, ORIGINS_LAT_COL,
    ORIGINS_X_COL, ORIGINS_Y_COL, PROJECT_CRS,
)
stops_gdf = gdf_from_csv_points(
    STOPS_CSV, STOP_ID_COL,
    STOP_LON_COL, STOP_LAT_COL,
    STOP_X_COL, STOP_Y_COL, PROJECT_CRS,
)
if STOP_FJ_COL not in stops_gdf.columns:
    raise ValueError(f"Stops file must include '{STOP_FJ_COL}' (arrivals/hour).")

stations_gdf = gdf_from_csv_points(
    STATIONS_CSV, STATION_ID_COL,
    STATION_LON_COL, STATION_LAT_COL,
    STATION_X_COL, STATION_Y_COL, PROJECT_CRS,
)

# --- Snap all points to nearest graph node ---
print("\nSnapping origins to walk graph...")
orig_gdf["walk_node"] = snap_points(orig_gdf, G_walk, "origins→walk")

print("Snapping stops to walk graph...")
stops_gdf["walk_node"] = snap_points(stops_gdf, G_walk, "stops→walk")

print("Snapping stops to bike graph...")
stops_gdf["bike_node"] = snap_points(stops_gdf, G_bike, "stops→bike")

print("Snapping stations to walk graph...")
stations_gdf["walk_node"] = snap_points(stations_gdf, G_walk, "stations→walk")

print("Snapping stations to bike graph...")
stations_gdf["bike_node"] = snap_points(stations_gdf, G_bike, "stations→bike")

print("\nDone snapping.")



Loading points...

Snapping origins to walk graph...
  [origins→walk]: snapped 2795 points.
Snapping stops to walk graph...
  [stops→walk]: snapped 2522 points.
Snapping stops to bike graph...
  [stops→bike]: snapped 2522 points.
Snapping stations to walk graph...
  [stations→walk]: snapped 290 points.
Snapping stations to bike graph...
  [stations→bike]: snapped 290 points.

Done snapping.


In [ ]:
# =========================
# 5) BUILD LOOKUPS
# =========================

# --- Stop lookups ---
# Walk-graph mapping: walk_node -> stop_ids
node_to_stops_walk = {}
# Bike-graph mapping: bike_node -> stop_ids  (fixes node-ID alignment issue)
node_to_stops_bike = {}
Fj = {}

for _, r in stops_gdf.iterrows():
    sid = str(r[STOP_ID_COL])
    Fj[sid] = float(r[STOP_FJ_COL])
    wn = int(r["walk_node"])
    bn = int(r["bike_node"])
    node_to_stops_walk.setdefault(wn, []).append(sid)
    node_to_stops_bike.setdefault(bn, []).append(sid)

# --- Station lookups ---
station_walk_node = dict(
    zip(stations_gdf[STATION_ID_COL].astype(str), stations_gdf["walk_node"].astype(int))
)
station_bike_node = dict(
    zip(stations_gdf[STATION_ID_COL].astype(str), stations_gdf["bike_node"].astype(int))
)

# walk_node -> station_ids (for determining which stations an origin can walk to)
walknode_to_station_ids = {}
for st_id, wn in station_walk_node.items():
    walknode_to_station_ids.setdefault(int(wn), set()).add(st_id)

# origin walk_node mapping
origin_walk_node = dict(
    zip(orig_gdf[ORIGINS_ID_COL].astype(str), orig_gdf["walk_node"].astype(int))
)

# --- Binary Fj for sensitivity (every stop counts as 1) ---
Fj_binary = {sid: 1.0 for sid in Fj}

print(f"Stops: {len(Fj)} | Stations: {len(station_bike_node)} | Origins: {len(origin_walk_node)}")
print("All good.")

Stops: 2522 | Stations: 290 | Origins: 2795
All good.


In [ ]:
# =========================
# 6) BASELINE SI COMPUTATION (walk=8, bike=12, frequency-weighted)
# =========================

print(f"Caching station bike reach (cutoff={BIKE_CUTOFF} min)...")
t0 = time.time()
station_bike_reach = cache_station_bike_reach(
    station_bike_node, G_bike, BIKE_CUTOFF, node_to_stops_bike
)
print(f"Bike cache time: {time.time() - t0:.1f}s")

print(f"\nComputing SI for {len(origin_walk_node)} origins (walk={WALK_CUTOFF}, bike={BIKE_CUTOFF})...")
t1 = time.time()
out = compute_si_for_origins(
    origin_walk_node, G_walk, WALK_CUTOFF,
    node_to_stops_walk, walknode_to_station_ids,
    station_bike_reach, Fj,
)
print(f"Origin loop time: {time.time() - t1:.1f}s")

OUT_CSV = os.path.join(CACHE_DIR, "SI_results.csv")
out.to_csv(OUT_CSV, index=False)
print(f"\nSaved: {OUT_CSV}")

Caching station bike reach (cutoff=12 min)...
Bike cache time: 5.3s

Computing SI for 2795 origins (walk=8, bike=12)...
Origin loop time: 0.4s

Saved: /Users/fgsumer/Desktop/thesis_code/Phase 1/cache/SI_results.csv


In [ ]:
print(out[["SI_walk", "SI_bike_added", "SI_total"]].describe())
print(f"\nShare benefiting from bike-transit: {(out['SI_bike_added'] > 0).mean():.3f}")
out.head(20)

           SI_walk  SI_bike_added     SI_total
count  2795.000000    2795.000000  2795.000000
mean      9.390340      63.314132    72.704472
std      27.686335     421.167334   441.945843
min       0.000000       0.000000     0.000000
25%       0.000000       0.000000     0.000000
50%       0.500000       0.000000     1.000000
75%       9.000000       0.000000     9.000000
max     534.500000    5814.500000  6349.000000

Share benefiting from bike-transit: 0.044


,origin_id,SI_walk,SI_bike_added,SI_total
0,CRS3035RES1000mN2933000E4032000,15.0,0.0,15.0
1,CRS3035RES1000mN2933000E4033000,22.0,0.0,22.0
2,CRS3035RES1000mN2933000E4034000,0.0,0.0,0.0
3,CRS3035RES1000mN2933000E4057000,0.0,0.0,0.0
4,CRS3035RES1000mN2934000E4029000,0.0,0.0,0.0
5,CRS3035RES1000mN2934000E4030000,0.0,0.0,0.0
6,CRS3035RES1000mN2934000E4031000,0.0,0.0,0.0
7,CRS3035RES1000mN2934000E4032000,0.0,0.0,0.0
8,CRS3035RES1000mN2934000E4033000,12.0,0.0,12.0
9,CRS3035RES1000mN2934000E4034000,0.0,0.0,0.0


In [ ]:
(out["SI_total"] == 0).sum() 

np.int64(1371)

In [ ]:
# Top 10 origins by SI_bike_added
top = out.nlargest(10, "SI_bike_added")[["origin_id", "SI_walk", "SI_bike_added", "SI_total"]]
print("\nTop 10 origins by SI_bike_added:")
print(top)


Top 10 origins by SI_bike_added:
                           origin_id  SI_walk  SI_bike_added  SI_total
550  CRS3035RES1000mN2950000E4041000    534.5         5814.5    6349.0
591  CRS3035RES1000mN2951000E4040000    448.0         5711.5    6159.5
592  CRS3035RES1000mN2951000E4041000    441.0         5596.5    6037.5
549  CRS3035RES1000mN2950000E4040000    280.5         5554.0    5834.5
632  CRS3035RES1000mN2952000E4040000    186.5         5381.0    5567.5
634  CRS3035RES1000mN2952000E4042000    157.0         5102.5    5259.5
590  CRS3035RES1000mN2951000E4039000    162.0         5054.5    5216.5
548  CRS3035RES1000mN2950000E4039000    153.0         4933.5    5086.5
507  CRS3035RES1000mN2949000E4040000    254.5         4490.0    4744.5
509  CRS3035RES1000mN2949000E4042000    161.0         4414.5    4575.5


In [ ]:
# =========================
# 7a) SENSITIVITY: THRESHOLD SCENARIOS (frequency-weighted Fj)
# =========================
# 3x3 grid: walk cutoff x bike cutoff


WALK_CUTOFFS = [5, 8, 12]
BIKE_CUTOFFS = [8, 12, 16]

out_dir = Path(CACHE_DIR) / "sensitivity_thresholds"
out_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []

for bike_cut in BIKE_CUTOFFS:
    print(f"\n--- Bike cutoff = {bike_cut} min ---")
    print("Caching station bike reach...")
    t0 = time.time()
    sbr = cache_station_bike_reach(station_bike_node, G_bike, bike_cut, node_to_stops_bike)
    print(f"  Cache time: {time.time() - t0:.1f}s")

    for walk_cut in WALK_CUTOFFS:
        label = f"w{walk_cut}_b{bike_cut}"
        print(f"  Computing SI: walk={walk_cut}, bike={bike_cut} ...")
        t1 = time.time()

        df = compute_si_for_origins(
            origin_walk_node, G_walk, walk_cut,
            node_to_stops_walk, walknode_to_station_ids,
            sbr, Fj,
        )
        df.to_csv(out_dir / f"SI_{label}.csv", index=False)

        benefited = (df["SI_bike_added"] > 0).sum()
        summary_rows.append({
            "walk_cutoff_min": walk_cut,
            "bike_cutoff_min": bike_cut,
            "fj_mode": "frequency",
            "n_origins": len(df),
            "mean_SI_walk": df["SI_walk"].mean(),
            "median_SI_walk": df["SI_walk"].median(),
            "mean_SI_bike_added": df["SI_bike_added"].mean(),
            "median_SI_bike_added": df["SI_bike_added"].median(),
            "mean_SI_total": df["SI_total"].mean(),
            "median_SI_total": df["SI_total"].median(),
            "share_benefited": benefited / max(len(df), 1),
        })

summary_freq = pd.DataFrame(summary_rows)
summary_freq.to_csv(out_dir / "sensitivity_summary_frequency.csv", index=False)
print("\nFrequency sensitivity summary saved.")



--- Bike cutoff = 8 min ---
Caching station bike reach...
  Cache time: 2.7s
  Computing SI: walk=5, bike=8 ...
  Computing SI: walk=8, bike=8 ...
  Computing SI: walk=12, bike=8 ...

--- Bike cutoff = 12 min ---
Caching station bike reach...
  Cache time: 5.6s
  Computing SI: walk=5, bike=12 ...
  Computing SI: walk=8, bike=12 ...
  Computing SI: walk=12, bike=12 ...

--- Bike cutoff = 16 min ---
Caching station bike reach...
  Cache time: 8.3s
  Computing SI: walk=5, bike=16 ...
  Computing SI: walk=8, bike=16 ...
  Computing SI: walk=12, bike=16 ...

Frequency sensitivity summary saved.


In [ ]:
# =========================
# 7b) SENSITIVITY: BINARY Fj (every stop = 1)
# =========================
# Same 3x3 threshold grid, but with Fj_binary instead of frequency-based Fj.
# Tests whether results are driven by the spatial pattern of stops
# or by a few high-frequency stops dominating the index.

out_dir_bin = Path(CACHE_DIR) / "sensitivity_binary"
out_dir_bin.mkdir(parents=True, exist_ok=True)

summary_rows_bin = []

for bike_cut in BIKE_CUTOFFS:
    print(f"\n--- Bike cutoff = {bike_cut} min (binary Fj) ---")
    t0 = time.time()
    sbr = cache_station_bike_reach(station_bike_node, G_bike, bike_cut, node_to_stops_bike)
    print(f"  Cache time: {time.time() - t0:.1f}s")

    for walk_cut in WALK_CUTOFFS:
        label = f"bin_w{walk_cut}_b{bike_cut}"
        print(f"  Computing binary SI: walk={walk_cut}, bike={bike_cut} ...")
        t1 = time.time()

        df = compute_si_for_origins(
            origin_walk_node, G_walk, walk_cut,
            node_to_stops_walk, walknode_to_station_ids,
            sbr, Fj_binary,
        )
        df.to_csv(out_dir_bin / f"SI_{label}.csv", index=False)

        benefited = (df["SI_bike_added"] > 0).sum()
        summary_rows_bin.append({
            "walk_cutoff_min": walk_cut,
            "bike_cutoff_min": bike_cut,
            "fj_mode": "binary",
            "n_origins": len(df),
            "mean_SI_walk": df["SI_walk"].mean(),
            "median_SI_walk": df["SI_walk"].median(),
            "mean_SI_bike_added": df["SI_bike_added"].mean(),
            "median_SI_bike_added": df["SI_bike_added"].median(),
            "mean_SI_total": df["SI_total"].mean(),
            "median_SI_total": df["SI_total"].median(),
            "share_benefited": benefited / max(len(df), 1),
        })
        

summary_bin = pd.DataFrame(summary_rows_bin)
summary_bin.to_csv(out_dir_bin / "sensitivity_summary_binary.csv", index=False)
print("\nBinary sensitivity summary saved.")



--- Bike cutoff = 8 min (binary Fj) ---
  Cache time: 2.7s
  Computing binary SI: walk=5, bike=8 ...
  Computing binary SI: walk=8, bike=8 ...
  Computing binary SI: walk=12, bike=8 ...

--- Bike cutoff = 12 min (binary Fj) ---
  Cache time: 5.4s
  Computing binary SI: walk=5, bike=12 ...
  Computing binary SI: walk=8, bike=12 ...
  Computing binary SI: walk=12, bike=12 ...

--- Bike cutoff = 16 min (binary Fj) ---
  Cache time: 8.5s
  Computing binary SI: walk=5, bike=16 ...
  Computing binary SI: walk=8, bike=16 ...
  Computing binary SI: walk=12, bike=16 ...

Binary sensitivity summary saved.


In [ ]:
# =========================
# 8) COMBINED SENSITIVITY OVERVIEW
# =========================

combined = pd.concat([summary_freq, summary_bin], ignore_index=True)
combined.to_csv(Path(CACHE_DIR) / "sensitivity_combined_summary.csv", index=False)
print("Combined summary saved.\n")
print(combined.to_string(index=False))

Combined summary saved.

 walk_cutoff_min  bike_cutoff_min   fj_mode  n_origins  mean_SI_walk  median_SI_walk  mean_SI_bike_added  median_SI_bike_added  mean_SI_total  median_SI_total  share_benefited
               5                8 frequency       2795      4.044186             0.0           18.469052                   0.0      22.513238              0.0         0.024687
               8                8 frequency       2795      9.390340             0.5           31.240608                   0.0      40.630948              1.0         0.043649
              12                8 frequency       2795     18.740966             5.5           44.501431                   0.0      63.242397              5.5         0.065474
               5               12 frequency       2795      4.044186             0.0           36.385868                   0.0      40.430054              0.0         0.024687
               8               12 frequency       2795      9.390340             0.5           


## How many residents fall into each accessibility class? 


In [ ]:

# ── CONFIG ───────────────────────────────────────────────────────────────
ORIGINS_PATH = str(DATA_DIR / "origins.csv")

SI_PATH = str(CACHE_DIR / "SI_results.csv") 



# SI columns to classify
ACCESS_COLS = {
    "SI_total": "total",
    "SI_bike_added": "gain"
}

# ── HELPERS ──────────────────────────────────────────────────────────────
# For suppressed population '<30', a midpoint approximation of 15 inhabitants used.

def parse_pop_value(x, gt = 15):
    """
    Convert Pop_grids values to numeric.
    Handles:
      - numeric values
      - strings like '<30', ' <30 ', '<=30'
    """
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    s = str(x).strip()

    # exact numeric string
    try:
        return float(s)
    except ValueError:
        pass

    # patterns like <30, <=30, < 30
    if re.match(r"^\s*<=?\s*30\s*$", s):
        return float(gt)

    # generic fallback: extract number if present
    m = re.search(r"(\d+(\.\d+)?)", s)
    if m:
        return float(m.group(1))

    return np.nan


def classify_accessibility(series, no_label="No access", low_label="Low",
                           med_label="Medium", high_label="High"):
    """
    Classification:
      - No access: value == 0
      - Low:      0 < value <= P25 of positive values
      - Medium:   P25 < value <= P75 of positive values
      - High:     value > P75 of positive values

    Percentiles are computed only among positive values.
    """
    s = pd.to_numeric(series, errors="coerce").fillna(0).copy()
    pos = s[s > 0]

    out = pd.Series(index=s.index, dtype="object")

    if len(pos) == 0:
        out[:] = no_label
        return out, np.nan, np.nan

    p25 = pos.quantile(0.25)
    p75 = pos.quantile(0.75)

    out[s == 0] = no_label
    out[(s > 0) & (s <= p25)] = low_label
    out[(s > p25) & (s <= p75)] = med_label
    out[s > p75] = high_label

    return out, p25, p75


def summarize_population_by_class(df, class_col, pop_col="Pop_grids_num", id_col="origin_id"):
    """
    Summary of population and number of cells by class.
    """
    d = df[[class_col, pop_col, id_col]].dropna(subset=[class_col]).copy()

    summary = (
        d.groupby(class_col, dropna=False)
         .agg(
             population=(pop_col, "sum"),
             n_cells=(id_col, "count")
         )
         .reset_index()
         .rename(columns={class_col: "class"})
    )

    total_pop = summary["population"].sum()
    total_cells = summary["n_cells"].sum()

    summary["pop_share_pct"] = np.where(total_pop > 0, 100 * summary["population"] / total_pop, np.nan)
    summary["cell_share_pct"] = np.where(total_cells > 0, 100 * summary["n_cells"] / total_cells, np.nan)

    class_order = ["No access", "Low", "Medium", "High"]
    summary["class"] = pd.Categorical(summary["class"], categories=class_order, ordered=True)
    summary = summary.sort_values("class").reset_index(drop=True)

    return summary

print("All good")


All good


In [ ]:
# ── LOAD DATA ────────────────────────────────────────────────────────────
orig = pd.read_csv(ORIGINS_PATH)
si = pd.read_csv(SI_PATH)

# Check required columns
if "GRD_ID" not in orig.columns:
    raise ValueError("origins.csv must contain column 'GRD_ID'") 
if "Pop_grids" not in orig.columns:
    raise ValueError("origins.csv must contain column 'Pop_grids'")

required_si = ["origin_id", "SI_walk", "SI_bike_added", "SI_total"]
missing_si = [c for c in required_si if c not in si.columns]
if missing_si:
    raise ValueError(f"SI_results.csv is missing required columns: {missing_si}")

# ── CLEAN POPULATION ─────────────────────────────────────────────────────
orig["Pop_grids_num"] = orig["Pop_grids"].apply(parse_pop_value)

print(f"Total cleaned population: {orig['Pop_grids_num'].sum():,.0f}")
print(f"Cells with missing population: {orig['Pop_grids_num'].isna().sum()}")



Total cleaned population: 646,170
Cells with missing population: 0


In [ ]:
# ── MERGE ────────────────────────────────────────────────────────────────
df = si.merge(
    orig[["GRD_ID", "Pop_grids", "Pop_grids_num"]],
    left_on="origin_id",
    right_on="GRD_ID",
    how="left"
)

if df["Pop_grids_num"].isna().any():
    print(f"Warning: {df['Pop_grids_num'].isna().sum()} SI rows have missing population after merge.")

# Keep only rows with valid positive population for summaries
df_valid = df[df["Pop_grids_num"].notna() & (df["Pop_grids_num"] > 0)].copy()

print(f"Rows with valid population: {len(df_valid):,}")
print(f"Population after merge: {df_valid['Pop_grids_num'].sum():,.0f}")

# ── CLASSIFY SI_total ────────────────────────────────────────────────────
df_valid["SI_total_class"], p25_total, p75_total = classify_accessibility(df_valid["SI_total"])
summary_total = summarize_population_by_class(df_valid, "SI_total_class")
print("\n=== SI_total classification ===")
print(summary_total.to_string(index=False))


# ── CLASSIFY SI_bike_added ───────────────────────────────────────────────
df_valid["SI_bike_added_class"], p25_gain, p75_gain = classify_accessibility(df_valid["SI_bike_added"])
summary_gain = summarize_population_by_class(df_valid, "SI_bike_added_class")
print("\n=== SI_bike_added classification ===")
print(summary_gain.to_string(index=False))



Rows with valid population: 1,636
Population after merge: 646,170

=== SI_total classification ===
    class  population  n_cells  pop_share_pct  cell_share_pct
No access     65649.0      566      10.159710       34.596577
      Low     35604.0      296       5.510005       18.092910
   Medium    130222.0      512      20.152901       31.295844
     High    414695.0      262      64.177384       16.014670

=== SI_bike_added classification ===
    class  population  n_cells  pop_share_pct  cell_share_pct
No access    376989.0     1521      58.342077       92.970660
      Low     39260.0       29       6.075800        1.772616
   Medium    117975.0       57      18.257579        3.484108
     High    111946.0       29      17.324543        1.772616


In [ ]:

df_valid.to_csv(str(CACHE_DIR / "accessibility_classes_for_pop.csv"), index=False)
print(f"Saved: {CACHE_DIR / 'accessibility_classes_for_population.csv'}")


Saved: /Users/fgsumer/Desktop/thesis_code/Phase 1/cache/accessibility_classes_for_population.csv


In [ ]:
df_valid.head()

,origin_id,SI_walk,SI_bike_added,SI_total,GRD_ID,Pop_grids,Pop_grids_num,SI_total_class,SI_bike_added_class
7,CRS3035RES1000mN2934000E4032000,0.0,0.0,0.0,CRS3035RES1000mN2934000E4032000,113,113.0,No access,No access
8,CRS3035RES1000mN2934000E4033000,12.0,0.0,12.0,CRS3035RES1000mN2934000E4033000,1107,1107.0,Medium,No access
10,CRS3035RES1000mN2934000E4036000,5.0,0.0,5.0,CRS3035RES1000mN2934000E4036000,<30,15.0,Low,No access
14,CRS3035RES1000mN2934000E4056000,5.0,0.0,5.0,CRS3035RES1000mN2934000E4056000,110,110.0,Low,No access
15,CRS3035RES1000mN2934000E4057000,0.0,0.0,0.0,CRS3035RES1000mN2934000E4057000,<30,15.0,No access,No access
